# What does R4 cost to run on a CPU?

On 24 November 2026, NLLB with its merged R4 digits adapter and the original Gemma 3 1B-IT translated the same 64 messages on a single Google Colab Xeon. The question was whether R4 would keep both times in a scenario correct after conversion to INT8, and what that would cost in latency, memory and disk space.

## Read the Colab CPU benchmark

**Run all** to recalculate the saved measurements using Python's standard library. The published outputs below come from that replay. The optional final section runs the original measurement protocol in a fresh Linux CPU process for each selected model. The model files are acquired separately. [Public run record](https://github.com/Silveroboros-dev/slavic-english-slm/blob/main/results/norm64_cpu_int8.json) · [Interactive comparison](https://silveroboros-dev.github.io/slavic-english-slm/#cpu-int8)

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

root = next(
    (
        p
        for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "slavic-english-slm")
        if (p / "results/norm64_run.json").is_file()
    ),
    None,
)
if root is None:
    root = Path.cwd() / "slavic-english-slm"
    subprocess.run(
        [
            "git",
            "clone",
            "--quiet",
            "--depth",
            "1",
            "https://github.com/Silveroboros-dev/slavic-english-slm.git",
            str(root),
        ],
        check=True,
    )
sys.path.insert(0, str(root))
from pipeline.build_norm64_dashboard import load_cpu_int8

case_path = root / "results/norm64_cases.jsonl"
case_bytes = case_path.read_bytes()
panel_hash = hashlib.sha256(case_bytes).hexdigest()
cases = [json.loads(line) for line in case_bytes.splitlines()]
record_path = root / "results/norm64_cpu_int8.json"
record = json.loads(record_path.read_text())
summary = load_cpu_int8(record_path, panel_hash, cases)["systems"]
settings = record["settings"]
print(record["date"], settings["cpu"], settings["intra_threads"], "logical CPUs")
print(
    "CTranslate2",
    settings["ctranslate2"],
    settings["actual_compute_type"],
    "| beam",
    settings["beam_size"],
    "| batch",
    settings["batch_size"],
)

2026-11-24 Intel(R) Xeon(R) CPU @ 2.20GHz 2 logical CPUs
CTranslate2 4.9.1 int8_float32 | beam 1 | batch 1


## Recalculate clock accuracy and resource use
The 48 Polish, Croatian and Russian messages form 24 pairs. The other 16 messages form eight Ukrainian pairs. Each pair passes only if blinded readings recover the expected clock from **both** outputs. Ukrainian received zero fine-tuning pairs and has its own normalization rule.

Median and nearest-rank p95 are computed from 64 individually timed requests per model after one warm-up. Each measurement includes tokenization, translation and decoding. Peak resident memory is for the entire process. Package size includes the converted weights, tokenizer and metadata.

In [2]:
names = {"nllb_r4": "NLLB + R4 digits", "gemma_1b": "Gemma 3 1B-IT"}
print(
    f"{'Archived model':18}  {'PL/HR/RU':>8}  {'UK':>3}  {'Median':>8}  {'p95':>8}  {'Peak RSS':>10}  {'Package':>12}"
)
for key, name in names.items():
    m = summary[key]
    print(
        f"{name:18}  {m['primary_passed']:5}/24  {m['uk_passed']}/8"
        f"  {m['median_latency_ms'] / 1000:6.2f} s  {m['p95_latency_ms'] / 1000:6.2f} s"
        f"  {m['peak_process_rss_bytes'] / 2**30:6.2f} GiB  {m['package_bytes'] / 1e6:8,.1f} MB"
    )

Archived model      PL/HR/RU   UK    Median       p95    Peak RSS       Package
NLLB + R4 digits       22/24  8/8    2.59 s    3.32 s    1.47 GiB     665.7 MB
Gemma 3 1B-IT          19/24  7/8    3.96 s    4.74 s    2.55 GiB   1,047.9 MB


## Which pairs passed in only one run?
The same total can hide a changed decision. The next cell compares per-pair outcomes against the archived full-precision run. These runs also used a different inference backend. A comparison between them cannot isolate quantization alone.

In [3]:
from collections import defaultdict
from pipeline.score_norm64 import correct

pairs = defaultdict(list)
for case in cases:
    pairs[case["pair_id"]].append(case)
for key, archived_arm in (("nllb_r4", "r4_normalized"), ("gemma_1b", "gemma_1b")):
    failed = set(record["systems"][key]["failed_clock_ids"])
    before = {pair for pair, rows in pairs.items() if all(correct(r, archived_arm) for r in rows)}
    after = {pair for pair, rows in pairs.items() if all(r["id"] not in failed for r in rows)}
    print(
        names[key],
        "| newly correct:",
        sorted(after - before),
        "| newly failed:",
        sorted(before - after),
    )

NLLB + R4 digits | newly correct: ['R4H-PL-04'] | newly failed: ['R4H-HR-02']
Gemma 3 1B-IT | newly correct: ['R4H-PL-04'] | newly failed: []


## Optional · generate and time fresh translations
Use a **Linux CPU** runtime with Python 3.11+ and compatible PyTorch. Set `fresh_systems` below to select one or both model IDs. Selecting a model installs the pinned conversion dependencies and reads or downloads several GB of weights. For Gemma, provide an authorized local copy of the original Kaggle Transformers checkpoint or use an account that already has access. For R4, point `r4_adapter` to the directory containing the matching adapter configuration and weights. The public repository provides their hashes. The frozen run used the original adapter; a new small training run produces different weights.

The converter merges the adapter into NLLB in float32 and quantizes the resulting weights. The benchmark pins each worker to the same two logical CPUs, loads its local tokenizer, translates one off-panel Polish message, then measures all 64 inputs in order. Raw results go to a new directory per run. [Conversion source](https://github.com/Silveroboros-dev/slavic-english-slm/blob/main/training/prepare_norm64_int8.py) · [Generation and measurement code](https://github.com/Silveroboros-dev/slavic-english-slm/blob/main/pipeline/bench_norm64_cpu.py)

In [4]:
import os
import tempfile

fresh_systems = ()  # e.g. ("gemma_1b",) or ("nllb_r4", "gemma_1b")
r4_adapter = None  # Path to the original R4 digits PEFT directory
gemma_checkpoint = None  # Optional local directory with the authorized original model

if fresh_systems:
    if len(set(fresh_systems)) != len(fresh_systems) or not set(fresh_systems) <= set(names):
        raise ValueError("Select each model at most once using the IDs shown above.")
    if (
        not sys.platform.startswith("linux")
        or len(os.sched_getaffinity(0)) < 2
        or sys.version_info < (3, 11)
    ):
        raise RuntimeError(
            "Select a Linux CPU runtime with two available logical CPUs and Python 3.11+."
        )
    if "nllb_r4" in fresh_systems and (r4_adapter is None or not Path(r4_adapter).is_dir()):
        raise ValueError("Supply the original R4 digits adapter directory for the NLLB run.")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "torch>=2.6",
            "ctranslate2==4.9.1",
            "transformers==4.57.1",
            "peft==0.17.1",
            "sentencepiece==0.2.1",
            "kagglehub==1.0.3",
            "accelerate==1.11.0",
        ],
        check=True,
    )
    workspace = Path.cwd() / "norm64-int8-work"
    workspace.mkdir(exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run-", dir=workspace))
    for key in fresh_systems:
        convert = [
            sys.executable,
            str(root / "training/prepare_norm64_int8.py"),
            "--system",
            key,
            "--work-dir",
            str(workspace),
        ]
        if key == "nllb_r4":
            convert.extend(["--adapter-dir", str(r4_adapter)])
        elif gemma_checkpoint is not None:
            convert.extend(["--gemma-dir", str(gemma_checkpoint)])
        subprocess.run(convert, check=True)
        subprocess.run(
            [
                sys.executable,
                str(root / "pipeline/bench_norm64_cpu.py"),
                "--system",
                key,
                "--model-dir",
                str(workspace / key),
                "--cases",
                str(case_path),
                "--output-dir",
                str(output),
                "--threads",
                "2",
            ],
            check=True,
        )
    print("Fresh output files:", output)

In [5]:
if fresh_systems:
    runs = [json.loads((output / (key + ".json")).read_bytes()) for key in fresh_systems]
    assert len({(r["vm_boot_id_sha256"], tuple(r["cpu_affinity"])) for r in runs}) == 1
    for r in runs:
        assert r["complete"] and r["case_file_sha256"] == panel_hash
        assert len({p["id"] for p in r["predictions"]}) == len(cases)
        matches = r["ct2_weight_sha256"] == record["systems"][r["system"]]["ct2_weight_sha256"]
        print(
            names[r["system"]],
            "| identical converted weights:",
            matches,
            "| median (s):",
            round(r["median_latency_ms"] / 1000, 2),
            "| peak RSS (GiB):",
            round(r["peak_process_rss_bytes"] / 2**30, 2),
        )
        print("First fresh English output:", r["predictions"][0]["translation"])

## How to interpret a new CPU run

The saved accuracy applies to the English texts judged in the original run. Give changed translations fresh, blinded clock readings before reporting a new accuracy. Timings from this CPU experiment describe the Colab machine. Phone measurements require a separate device run.